### Description
Loads cleaned parquets for Track B datasets (depmap_expr, hpa_rna, hpa_desc, geo_expr), verifies candidate gene and cell line keys against shared lookup tables, measures match/overlap rates, and surfaces unresolved or duplicate identities for handoff to 03_data_integration.


depmap_expr, hpa_rna, geo_expr --> all three need GENE key harmonisation (ENSG) and all three need CELL LINE key resolution

hpa_desc -- > exists specifically to help resolve hpa_rna's cell line identity gaps (it has CVCL accessions that hpa_rna itself lacks)

### Confirm gene key format across all three expression files

In [9]:
import pandas as pd
import re

depmap = pd.read_parquet("../../data/parquet/data_clean/depmap_expr_clean.parquet")
hpa    = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
geo    = pd.read_parquet("../../data/parquet/data_clean/geo_expr_clean.parquet")

# depmap_expr — gene is in COLUMN HEADERS, already cleaned by
# clean_depmap_expr() to extract bare ensg from "symbol (ENSGxxx)"
depmap_genes = set(c for c in depmap.columns if re.match(r"^ensg\d+$", str(c)))
print(f"depmap_expr: {len(depmap_genes)} clean ENSG columns")
print(f"depmap_expr: {len(depmap.columns) - len(depmap_genes)} columns did NOT match clean ENSG pattern")
non_ensg_cols = [c for c in depmap.columns if c not in depmap_genes]
print(f"Sample non-ENSG columns: {non_ensg_cols[:10]}")

# hpa_rna — gene is a ROW VALUE
hpa_genes = set(hpa["gene"].dropna())
bad_hpa = hpa[~hpa["gene"].astype(str).str.match(r"^ensg\d+$", na=False)]
print(f"\nhpa_rna: {len(hpa_genes)} unique gene values")
print(f"hpa_rna: {len(bad_hpa)} rows with non-clean ENSG format")

# geo_expr — gene is also a ROW VALUE (still wide, gene per row)
geo_genes = set(geo["gene"].dropna())
bad_geo = geo[~geo["gene"].astype(str).str.match(r"^ensg\d+$", na=False)]
print(f"\ngeo_expr: {len(geo_genes)} unique gene values")
print(f"geo_expr: {len(bad_geo)} rows with non-clean ENSG format")

depmap_expr: 53961 clean ENSG columns
depmap_expr: 0 columns did NOT match clean ENSG pattern
Sample non-ENSG columns: []

hpa_rna: 20162 unique gene values
hpa_rna: 0 rows with non-clean ENSG format

geo_expr: 19914 unique gene values
geo_expr: 0 rows with non-clean ENSG format


### Gene overlap across all three

In [12]:
print(f"depmap ∩ hpa:        {len(depmap_genes & hpa_genes)}")
print(f"depmap ∩ geo:         {len(depmap_genes & geo_genes)}")
print(f"hpa ∩ geo:            {len(hpa_genes & geo_genes)}")
print(f"all three:            {len(depmap_genes & hpa_genes & geo_genes)}")
print(f"depmap only:          {len(depmap_genes - hpa_genes - geo_genes)}")
print(f"hpa only:             {len(hpa_genes - depmap_genes - geo_genes)}")
print(f"geo only:              {len(geo_genes - depmap_genes - hpa_genes)}")

depmap ∩ hpa:        19896
depmap ∩ geo:         19894
hpa ∩ geo:            16062
all three:            16060
depmap only:          30231
hpa only:             264
geo only:              18


### Cell line key resolution for depmap_expr

In [15]:
depmap_profiles = pd.read_parquet("../../data/parquet/data_clean/depmap_profiles_clean.parquet")
sample_info     = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

# depmap_expr's row index is PR- ProfileID (per clean_depmap_expr,
# the index was lowercased but not otherwise transformed)
print(depmap.index[:5].tolist())

# Check PR- IDs in depmap_expr actually exist in depmap_profiles
depmap_pr_ids = set(depmap.index)
profile_pr_ids = set(depmap_profiles["profileid"].dropna())

print(f"depmap_expr PR- IDs:    {len(depmap_pr_ids)}")
print(f"In depmap_profiles too: {len(depmap_pr_ids & profile_pr_ids)}")
print(f"Missing from bridge:    {len(depmap_pr_ids - profile_pr_ids)}")

# Filter depmap_profiles to RNA datatype only, then bridge to ACH-
rna_profiles = depmap_profiles[depmap_profiles["datatype"] == "rna"]
pr_to_ach = dict(zip(rna_profiles["profileid"], rna_profiles["modelid"]))

depmap_ach = depmap.index.map(pr_to_ach)
print(f"Resolved to ACH-: {depmap_ach.notna().sum()} / {len(depmap)}")

# Now confirm ACH- exists in sample_info — the final hop
ach_in_sample_info = set(sample_info["depmap_id"].dropna())
resolved_ach = set(depmap_ach.dropna())
print(f"ACH- found in sample_info: {len(resolved_ach & ach_in_sample_info)} / {len(resolved_ach)}")

['pr-adbjpg', 'pr-i2azwg', 'pr-5ekaac', 'pr-i21681', 'pr-i9drp1']
depmap_expr PR- IDs:    1495
In depmap_profiles too: 1495
Missing from bridge:    0
Resolved to ACH-: 1495 / 1495
ACH- found in sample_info: 1412 / 1479


In [25]:
# Your orphaned depmap_expr cell lines (failed sample_info match)
orphaned_ach_ids = set(depmap_ach.dropna()) - set(sample_info["depmap_id"])


signatures     = pd.read_parquet("../../data/parquet/data_clean/signatures_clean.parquet")
# Check if they exist in the OTHER DepMap files instead
in_depmap_profiles = orphaned_ach_ids & set(depmap_profiles["modelid"])
in_signatures = orphaned_ach_ids & set(signatures["modelid"])  # if you have access

print(f"Orphaned against sample_info: {len(orphaned_ach_ids)}")
print(f"Of those, found in depmap_profiles: {len(in_depmap_profiles)}")
print(f"Of those, found in signatures: {len(in_signatures)}")

Orphaned against sample_info: 67
Of those, found in depmap_profiles: 67
Of those, found in signatures: 42


In [26]:
remaining_25 = orphaned_ach_ids - in_signatures
print(remaining_25)

# Check if signatures itself has a default-entry filter that
# might be excluding these legitimately (recall File 14 needed
# IsDefaultEntryForModel == "Yes" filtering before use)
print(signatures[signatures["modelid"].isin(remaining_25)])
# vs checking the unfiltered raw signatures count for these IDs

{'ach-003148', 'ach-003153', 'ach-003143', 'ach-003133', 'ach-003139', 'ach-003147', 'ach-003134', 'ach-003159', 'ach-003138', 'ach-003142', 'ach-003136', 'ach-003161', 'ach-003160', 'ach-003132', 'ach-003154', 'ach-003152', 'ach-003149', 'ach-003145', 'ach-003155', 'ach-003135', 'ach-003158', 'ach-003157', 'ach-003141', 'ach-003156', 'ach-003150'}
Empty DataFrame
Columns: [signature_index, sequencingid, modelid, modelconditionid, isdefaultentryformodel, isdefaultentryformc, msiscore, lohfraction, wgd, cin, ploidy, aneuploidy]
Index: []


In [27]:
# Check the full numeric ID range present in EACH file, not just
# the unresolved subset — this tells you if depmap_profiles
# generally extends further than sample_info and signatures

import re

def get_id_numbers(id_set):
    return sorted(int(re.search(r'\d+', x).group()) for x in id_set)

profiles_ids = get_id_numbers(set(depmap_profiles["modelid"]))
sample_info_ids = get_id_numbers(set(sample_info["depmap_id"]))
signatures_ids = get_id_numbers(set(signatures["modelid"]))

print(f"depmap_profiles max ID: ACH-{profiles_ids[-1]:06d}")
print(f"sample_info max ID:     ACH-{sample_info_ids[-1]:06d}")
print(f"signatures max ID:      ACH-{signatures_ids[-1]:06d}")

depmap_profiles max ID: ACH-003161
sample_info max ID:     ACH-002926
signatures max ID:      ACH-003480


In [28]:
log_entries = []
for ach_id in orphaned_ach_ids:
    if ach_id in in_signatures:
        reason = "verified_in_signatures_missing_from_sample_info"
    else:
        reason = "unresolvable_newer_than_sample_info_no_signature_data"
    log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

unmapped_log = pd.DataFrame(log_entries)
unmapped_log.to_csv("unmapped_depmap_expr.csv", index=False)

{Ellipsis}

In [29]:
# Are the 42 contiguous too, or scattered? This tells you whether
# they're one batch or several different historical additions
print(sorted(in_signatures))

['ach-001437', 'ach-001438', 'ach-001481', 'ach-001508', 'ach-001537', 'ach-001672', 'ach-001679', 'ach-001691', 'ach-001693', 'ach-001705', 'ach-001847', 'ach-001854', 'ach-001855', 'ach-001971', 'ach-001975', 'ach-001986', 'ach-001990', 'ach-002035', 'ach-002070', 'ach-002485', 'ach-002486', 'ach-002497', 'ach-002522', 'ach-002523', 'ach-002524', 'ach-002526', 'ach-002531', 'ach-002533', 'ach-002535', 'ach-002539', 'ach-002650', 'ach-002654', 'ach-002780', 'ach-002781', 'ach-002784', 'ach-002787', 'ach-002799', 'ach-002801', 'ach-002806', 'ach-002921', 'ach-002922', 'ach-002925']


In [30]:
cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")
# Check if Cellosaurus's cross-reference column contains any of your orphaned ACH- IDs
# (Cellosaurus stores DepMap IDs as cross-references in some entries)
orphan_check = cellosaurus[cellosaurus.apply(
    lambda row: any(oid in str(row.values) for oid in orphaned_ach_ids), axis=1
)]

In [31]:
print(orphan_check.shape)
print(len(orphan_check))

(26, 17)
26


In [32]:
# Step 1 — find the column that holds DepMap ACH- references
for col in cellosaurus.columns:
    sample = cellosaurus[col].dropna().astype(str)
    if sample.str.contains("ach-", case=False, na=False).any():
        print(f"Column '{col}' contains ACH- style values")
        print(sample[sample.str.contains("ach-", case=False, na=False)].head(3).tolist())

Column 'cellosaurus_cell_line_name' contains ACH- style values
['ach-1p', 'ach-2', 'ach-3p']
Column 'synonyms' contains ACH- style values
['mz-sto-1; mainz-stomach-1', 'rat glandular stomach-2', 'rat glandular stomach-3a']
Column 'cross-references' contains ACH- style values
['cancercelllines; cvcl_n588 || cell_model_passport; sidm01315 || cosmic; 2307723 || depmap; ach-001270 || wikidata; q54581310', 'bto; bto:0003605 || clo; clo_0001072 || mccl; mcc:0000001 || cldb; cl5 || biosample; samn03471026 || cancercelllines; cvcl_0110 || cell_model_passport; sidm01967 || depmap; ach-001000 || ecacc; 86030402 || geo; gsm827421 || geo; gsm886835 || geo; gsm887898 || geo; gsm1374373 || geo; gsm3034453 || geo; gsm3034454 || geo; gsm3034455 || geo; gsm3034456 || geo; gsm3034457 || geo; gsm3034458 || iarc_tp53; 21148 || izsler; bs tcl 109 || lincs_ldp; lcl-1393 || ncbi_iran; c118 || pharmacodb; 1321n1_2_2019 || progenetix; cvcl_0110 || wikidata; q54581478', 'bto; bto:0003167 || clo; clo_0001084 || 

In [33]:
import re

def extract_depmap_id(xref_string):
    if pd.isna(xref_string):
        return None
    match = re.search(r"depmap;\s*(ach-\d+)", str(xref_string))
    return match.group(1) if match else None

cellosaurus["depmap_id_extracted"] = cellosaurus["cross-references"].apply(extract_depmap_id)

# Now do a precise, real membership check
found_in_cellosaurus = cellosaurus[
    cellosaurus["depmap_id_extracted"].isin(orphaned_ach_ids)
]
print(f"Of {len(orphaned_ach_ids)} orphaned IDs, found in Cellosaurus cross-references: {len(found_in_cellosaurus)}")
print(found_in_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession", "depmap_id_extracted"]])

Of 67 orphaned IDs, found in Cellosaurus cross-references: 26
       cellosaurus_cell_line_name cellosaurus_accession depmap_id_extracted
1020                       1618-k             cvcl_s505          ach-001437
1151                  1777n rpmet             cvcl_2277          ach-001438
35361                     chla-90             cvcl_6610          ach-001481
51868                     etcc016             cvcl_y086          ach-002650
83313                       hkbmm             cvcl_8162          ach-002070
96003                      jve015             cvcl_eg16          ach-002654
97144                    kku-m055             cvcl_m258          ach-001537
99053                       lcam1             cvcl_e062          ach-002035
102365                    maver-1             cvcl_1831          ach-002485
104231                     mes-ov             cvcl_cz92          ach-002486
105425                      mm466             cvcl_4449          ach-001971
107672                ncc-

In [34]:
def extract_discontinued_status(comment_string):
    if pd.isna(comment_string):
        return None
    match = re.search(r"discontinued:\s*depmap;\s*(ach-\d+);\s*true", str(comment_string))
    return match.group(1) if match else None

cellosaurus["discontinued_depmap_id"] = cellosaurus["comments"].apply(extract_discontinued_status)

discontinued_ids = set(cellosaurus["discontinued_depmap_id"].dropna())

overlap_with_orphans = orphaned_ach_ids & discontinued_ids
print(f"Of my {len(orphaned_ach_ids)} orphaned IDs, confirmed DISCONTINUED by DepMap: {len(overlap_with_orphans)}")
print(overlap_with_orphans)

Of my 67 orphaned IDs, confirmed DISCONTINUED by DepMap: 0
set()


In [40]:
group_25 = remaining_25  # your tight-block IDs from earlier
group_42 = orphaned_ach_ids - group_25

found_set = set(found_in_cellosaurus["depmap_id_extracted"])

print(f"Of the 26 recovered, from group_25 (tight block): {len(found_set & group_25)}")
print(f"Of the 26 recovered, from group_42 (scattered):   {len(found_set & group_42)}")

Of the 26 recovered, from group_25 (tight block): 0
Of the 26 recovered, from group_42 (scattered):   26


In [41]:
print(len(group_25)), print(len(found_set))

25
26


(None, None)

In [39]:
log_entries = []
crosswalk_additions = []

for ach_id in orphaned_ach_ids:
    if ach_id in found_set:  # the 26 recovered via Cellosaurus cross-reference
        # Pull the matching row from your earlier Cellosaurus lookup
        match = found_in_cellosaurus[
            found_in_cellosaurus["depmap_id_extracted"] == ach_id
        ].iloc[0]

        crosswalk_additions.append({
            "model_id": ach_id,
            "cvcl_id": match["cellosaurus_accession"],
            "canonical_name": match["cellosaurus_cell_line_name"],
            "resolution_method": "cellosaurus_xref_fallback"
        })

    elif ach_id in group_25:  # tight block, confirmed absent from all 3 sources
        reason = "unresolvable_too_new_confirmed_3way"
        log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

    else:  # remaining 16 from group_42, real but missing from sample_info
        reason = "verified_real_missing_from_sample_info"
        log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

# Save the 41 genuinely unresolved
unmapped_log = pd.DataFrame(log_entries)
unmapped_log.to_csv("unmapped_depmap_expr.csv", index=False)
print(f"Unmapped log: {len(unmapped_log)} rows")  # should be 41

# Save the 26 newly resolved, ready to merge into your crosswalk
crosswalk_fallback = pd.DataFrame(crosswalk_additions)
crosswalk_fallback.to_csv("crosswalk_additions_cellosaurus_fallback.csv", index=False)
print(f"Crosswalk additions: {len(crosswalk_fallback)} rows")  # should be 26

Unmapped log: 41 rows
Crosswalk additions: 26 rows


Everything so far has been investigation (figuring out match rates, finding orphans, testing fallbacks)

### Now creating final depmap_expr cell line crosswalk

In [43]:
print(len(pr_to_ach))           # PR- → ACH- mapping dict, from depmap_profiles
print(len(orphaned_ach_ids))    # the 67
print(len(found_set))           # the 26 recovered via Cellosaurus
print(len(group_25))            # the 25 too-new
sample_info[["depmap_id", "rrid", "cell_line_name"]].head()
found_in_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession", "depmap_id_extracted"]].head()

1495
67
26
25


,cellosaurus_cell_line_name,cellosaurus_accession,depmap_id_extracted
1020,1618-k,cvcl_s505,ach-001437
1151,1777n rpmet,cvcl_2277,ach-001438
35361,chla-90,cvcl_6610,ach-001481
51868,etcc016,cvcl_y086,ach-002650
83313,hkbmm,cvcl_8162,ach-002070


### Building the full PR- → ACH- mapping table, not just the orphans

In [44]:
all_profile_ids = depmap.index  # every PR- ID in your depmap_expr table

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

print(crosswalk.shape)
print(crosswalk["model_id"].isna().sum())  # PR- IDs with no ACH- at all — different problem, check separately

(1495, 2)
0


### Handle the dual-profile issue 
We flagged earlier that some cell lines have two PR- IDs pointing to the same ACH-. Resolve this now, since it affects row counts downstream:

In [45]:
dupe_models = crosswalk["model_id"].value_counts()
dupes = dupe_models[dupe_models > 1]
print(f"Cell lines with multiple RNA profiles: {len(dupes)}")

# Decision: keep highest-coverage profile per model_id
# (using total expression sum as a coverage proxy, per earlier agreement)
coverage = depmap.sum(axis=1)  # sum across genes per PR- row
crosswalk["coverage"] = crosswalk["profile_id"].map(coverage)

crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
)
print(f"After dedup: {len(crosswalk)} rows")

Cell lines with multiple RNA profiles: 16
After dedup: 1479 rows


### Attaching identity for the cleanly resolved group (matched sample_info directly)

In [53]:
resolved_mask = crosswalk["model_id"].isin(sample_info["depmap_id"])

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

crosswalk["resolution_method"] = None
crosswalk.loc[resolved_mask, "resolution_method"] = "sample_info_direct"
crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

In [47]:
print(type(resolved_mask))
print(resolved_mask.shape)

print(crosswalk.columns.tolist())
print(crosswalk.columns[crosswalk.columns.duplicated()])  # any duplicate column names?

print(type(crosswalk["model_id"]))  # should say Series — if it says DataFrame, that's the bug

<class 'pandas.core.series.Series'>
(1479,)
['profile_id', 'model_id', 'coverage', 'depmap_id', 'rrid', 'cell_line_name', 'lineage', 'primary_disease', 'resolution_method']
Index([], dtype='object')
<class 'pandas.core.series.Series'>


In [55]:
print(resolved_mask.index[:10].tolist())
print(crosswalk.index[:10].tolist())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [54]:
crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
    .reset_index(drop=True)   # ← add this, fixes the root cause generally
)

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

# Compute the mask AFTER the merge, using a column that's actually
# part of crosswalk right now — guaranteed correct index alignment
crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"

crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

print(crosswalk["resolution_method"].value_counts(dropna=False))

KeyError: 'coverage'

In [56]:
crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"

print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"Total rows: {len(crosswalk)}")  # should be 1479

resolution_method
sample_info_direct    1428
None                    67
Name: count, dtype: int64
Total rows: 1495


In [57]:
all_profile_ids = depmap.index

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

coverage = depmap.sum(axis=1)
crosswalk["coverage"] = crosswalk["profile_id"].map(coverage)

crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
    .reset_index(drop=True)
)

print(f"After dedup: {len(crosswalk)} rows")  # should be 1479, not 1495

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"
crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"Total rows: {len(crosswalk)}")  # should now be 1479

After dedup: 1479 rows
resolution_method
sample_info_direct    1412
None                    67
Name: count, dtype: int64
Total rows: 1479


### Attaching identity for the 26 recovered via Cellosaurus fallback

In [58]:
fallback_mask = crosswalk["model_id"].isin(found_set)

for idx in crosswalk[fallback_mask].index:
    ach_id = crosswalk.loc[idx, "model_id"]
    match = found_in_cellosaurus[found_in_cellosaurus["depmap_id_extracted"] == ach_id]
    if len(match) > 0:
        crosswalk.loc[idx, "cvcl_id"] = match.iloc[0]["cellosaurus_accession"]
        crosswalk.loc[idx, "canonical_name"] = match.iloc[0]["cellosaurus_cell_line_name"]
        crosswalk.loc[idx, "resolution_method"] = "cellosaurus_xref_fallback"

### Marking the remaining 41 as explicitly unresolved (don't drop, don't fill blanks silently)

In [59]:
unresolved_mask = crosswalk["cvcl_id"].isna()
crosswalk.loc[unresolved_mask & crosswalk["model_id"].isin(group_25), "resolution_method"] = "unresolvable_too_new_confirmed_3way"
crosswalk.loc[unresolved_mask & ~crosswalk["model_id"].isin(group_25), "resolution_method"] = "verified_real_missing_from_sample_info"

print(crosswalk["resolution_method"].value_counts())

resolution_method
sample_info_direct                        1412
cellosaurus_xref_fallback                   26
unresolvable_too_new_confirmed_3way         25
verified_real_missing_from_sample_info      16
Name: count, dtype: int64


### Validating

In [60]:
assert crosswalk["model_id"].notna().all(), "Null model_id found!"
assert crosswalk["model_id"].is_unique, "Duplicate model_id found!"
print(f"Total rows: {len(crosswalk)}")
print(f"Resolved (any method): {crosswalk['cvcl_id'].notna().sum()}")
print(f"Unresolved: {crosswalk['cvcl_id'].isna().sum()}")

Total rows: 1479
Resolved (any method): 1438
Unresolved: 41


### Saving files

In [61]:
final_crosswalk = crosswalk[crosswalk["cvcl_id"].notna()][
    ["model_id", "cvcl_id", "canonical_name", "lineage", "primary_disease", "resolution_method"]
]
unmapped = crosswalk[crosswalk["cvcl_id"].isna()][
    ["model_id", "profile_id", "resolution_method"]
]

final_crosswalk.to_csv("depmap_expr_cell_line_crosswalk.csv", index=False)
unmapped.to_csv("unmapped_depmap_expr.csv", index=False)

print(f"Final crosswalk: {len(final_crosswalk)} rows")
print(f"Unmapped log: {len(unmapped)} rows")

Final crosswalk: 1438 rows
Unmapped log: 41 rows


In [51]:
all_profile_ids = depmap.index  # every PR- ID in your depmap_expr table

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

print(crosswalk.shape)
print(crosswalk["model_id"].isna().sum())  # PR- IDs with no ACH- at all — different problem, check separately

(1495, 2)
0


### HPA_RNA Resolution : Cell line

Resolution chain logic:
Tier 1: normalised hpa_rna name → exact match vs sample_info.cell_line_name
Tier 2: normalised hpa_rna name → match vs sample_info.stripped_cell_line_name
Tier 3: normalised hpa_rna name → look up CVCL via hpa_desc → match vs sample_info.rrid
Unresolved → log

### Running the corrected three-tier chain cleanly, end to end, and capture real numbers

In [ ]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
hpa      = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

import re
def normalise(name):
    if pd.isna(name):
        return None
    return re.sub(r"[-/_\s]", "", str(name).lower())

# Use ONE consistent column name everywhere — cln_norm
hpa["cln_norm"]         = hpa["cell line"].apply(normalise)
sample_info["cln_norm"] = sample_info["cell_line_name"].apply(normalise)
sample_info["scln_norm"] = sample_info["stripped_cell_line_name"].apply(normalise)
hpa_desc["cln_norm"]    = hpa_desc["cell line"].apply(normalise)
sample_info["rrid_norm"] = sample_info["rrid"]

tier1 = hpa["cln_norm"].isin(sample_info["cln_norm"])
print(f"Tier 1: {hpa.loc[tier1, 'cell line'].nunique()} unique cell lines")

remaining = hpa[~tier1].reset_index(drop=True)
tier2 = remaining["cln_norm"].isin(sample_info["scln_norm"])
print(f"Tier 2: {remaining.loc[tier2, 'cell line'].nunique()} additional")

still_remaining = remaining[~tier2].reset_index(drop=True)
merged = still_remaining.merge(
    hpa_desc[["cln_norm", "cellosaurus id"]], on="cln_norm", how="left"
)
tier3 = merged["cellosaurus id"].isin(sample_info["rrid_norm"])
print(f"Tier 3: {merged.loc[tier3, 'cell line'].nunique()} additional")

unresolved = merged[~tier3]
print(f"Unresolved: {unresolved['cell line'].nunique()}")

Tier 1: 1014 unique cell lines
Tier 2: 35 additional
Tier 3: 63 additional
Unresolved: 94


In [63]:
print(sorted(unresolved["cell line"].unique())[:20])
print(f"Total unresolved unique names: {unresolved['cell line'].nunique()}")

['537-mel', '624-mel', '888-mel', 'asc52telo', 'bewo', 'bj [human fibroblast]', 'bjab', 'c170', 'ccrf-sb', 'colo 206f', 'colo 320dm', 'colo 699', 'cor-l26', 'cov413b', 'deoc-1', 'fhdf/tert166', 'gtl-16', 'hacat', 'hbec3-kt', 'hbf/tert88']
Total unresolved unique names: 94


In [64]:
import re

def normalise(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)   # strip bracketed annotations
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

print(normalise("bj [human fibroblast]"))  # should now just be "bj"

bj


In [65]:
unresolved_names_norm = set(unresolved["cln_norm"].unique())  # after re-running with the fix

cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")
cellosaurus["name_norm"] = cellosaurus["cellosaurus_cell_line_name"].apply(normalise)

found_via_cellosaurus = cellosaurus[cellosaurus["name_norm"].isin(unresolved_names_norm)]
print(f"Recovered via direct Cellosaurus name match: {len(found_via_cellosaurus)}")
print(found_via_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

Recovered via direct Cellosaurus name match: 93
       cellosaurus_cell_line_name cellosaurus_accession
3941                      537-mel             cvcl_8052
4254                      624-mel             cvcl_8054
4953                      888-mel             cvcl_4632
16515                   asc52telo             cvcl_u602
29300                        bewo             cvcl_0044
...                           ...                   ...
143906                   u-266/84             cvcl_0016
143908                     u-2932             cvcl_1896
144923                  ucsd-242l             cvcl_m009
145565                      uke-1             cvcl_0104
149745                  wsu-fsccl             cvcl_1903

[93 rows x 2 columns]


In [66]:
cellosaurus_syn = cellosaurus.explode("synonyms") if cellosaurus["synonyms"].dtype == "object" else cellosaurus
# adjust based on actual structure — check first:
print(cellosaurus["synonyms"].dropna().iloc[0])  # see actual format before exploding

z48-5mg-70


In [67]:
fully_resolved = set(found_via_cellosaurus["cellosaurus_cell_line_name"])
still_unresolved = unresolved_names_norm - {normalise(n) for n in fully_resolved}
print(still_unresolved)

{'bj[humanfibroblast]'}


In [69]:
recovered_cvcls = set(found_via_cellosaurus["cellosaurus_accession"])
sample_info_cvcls = set(sample_info["rrid"].dropna())

has_ach = recovered_cvcls & sample_info_cvcls
no_ach  = recovered_cvcls - sample_info_cvcls

print(f"Of 93 recovered, have a matching ACH- in sample_info: {len(has_ach)}")
print(f"Of 93 recovered, CVCL exists but NO ACH- equivalent:  {len(no_ach)}")

Of 93 recovered, have a matching ACH- in sample_info: 0
Of 93 recovered, CVCL exists but NO ACH- equivalent:  93


In [68]:
import re

def normalise(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)   # strip bracketed annotations
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

test = normalise("bj [human fibroblast]")
print(test)  # should print "bj"

# Check if "bj" resolves against sample_info directly
print(sample_info[sample_info["cln_norm"].apply(lambda x: normalise(x) if pd.notna(x) else x) == test].shape)

# Or more directly, since sample_info's cln_norm was built with the OLD
# normalise function, just check the raw cell_line_name column
print(sample_info[sample_info["cell_line_name"] == "bj"])

bj
(0, 32)
Empty DataFrame
Columns: [depmap_id, cell_line_name, stripped_cell_line_name, ccle_name, alias, cosmicid, sex, source, rrid, wtsi_master_cell_id, sample_collection_site, primary_or_metastasis, primary_disease, subtype, age, sanger_model_id, depmap_public_comments, lineage, lineage_subtype, lineage_sub_subtype, lineage_molecular_subtype, default_growth_pattern, model_manipulation, model_manipulation_details, patient_id, parent_depmap_id, cellosaurus_ncit_disease, cellosaurus_ncit_id, cellosaurus_issues, cln_norm, scln_norm, rrid_norm]
Index: []

[0 rows x 32 columns]


In [70]:
# Search loosely rather than requiring exact match
candidates = sample_info[sample_info["cell_line_name"].str.contains("bj", case=False, na=False)]
print(candidates[["cell_line_name", "stripped_cell_line_name", "alias", "rrid"]])

# Also check alias column directly, since BJ is a common enough name
# it might appear as an alias on a differently-named row
alias_candidates = sample_info[sample_info["alias"].str.contains("bj", case=False, na=False)]
print(alias_candidates[["cell_line_name", "alias", "rrid"]])

     cell_line_name stripped_cell_line_name alias       rrid
111        bj htert                 bjhtert  none  cvcl_6573
1541       va-es-bj                  vaesbj  none  cvcl_1785
Empty DataFrame
Columns: [cell_line_name, alias, rrid]
Index: []


In [71]:
test_norm = normalise("bj [human fibroblast]")  # now "bj" with the fixed function
print(test_norm)

found_bj = cellosaurus[cellosaurus["name_norm"] == test_norm]
print(found_bj[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

bj
                 cellosaurus_cell_line_name cellosaurus_accession
29746                 bj [human b-cell ihw]             cvcl_e483
29747                 bj [human fibroblast]             cvcl_3653
29748  bj [human pancreatic adenocarcinoma]             cvcl_li18


In [72]:
if len(found_bj) > 0:
    bj_cvcl = found_bj.iloc[0]["cellosaurus_accession"]
    print(f"BJ's CVCL: {bj_cvcl}")
    ach_match = sample_info[sample_info["rrid"] == bj_cvcl]
    print(ach_match[["depmap_id", "cell_line_name", "rrid"]])

BJ's CVCL: cvcl_e483
Empty DataFrame
Columns: [depmap_id, cell_line_name, rrid]
Index: []


In [73]:
def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower() if match else None

hpa_bracket_content = get_bracket_content("bj [human fibroblast]")
print(hpa_bracket_content)  # "human fibroblast"

# Now match against Cellosaurus's bracket content too, not just the base name
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

correct_match = cellosaurus[
    (cellosaurus["name_norm"] == "bj") &
    (cellosaurus["bracket_content"] == hpa_bracket_content)
]
print(correct_match[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

human fibroblast
      cellosaurus_cell_line_name cellosaurus_accession
29747      bj [human fibroblast]             cvcl_3653


In [74]:
correct_cvcl = "cvcl_3653"
ach_match = sample_info[sample_info["rrid"] == correct_cvcl]
print(ach_match[["depmap_id", "cell_line_name", "rrid"]])

Empty DataFrame
Columns: [depmap_id, cell_line_name, rrid]
Index: []


In [75]:
# Check if ANY base name in Cellosaurus has multiple bracket variants
cellosaurus_bracketed = cellosaurus[cellosaurus["cellosaurus_cell_line_name"].str.contains(r"\[", na=False)]
collision_check = cellosaurus_bracketed.groupby("name_norm").size()
collisions = collision_check[collision_check > 1]
print(f"Base names with multiple bracket-variant entries in Cellosaurus: {len(collisions)}")
print(collisions)

Base names with multiple bracket-variant entries in Cellosaurus: 730
name_norm
10a6     3
10c9     4
10e2     2
10f7     2
110      2
        ..
xp1ch    2
xp1sa    3
xp1se    2
xp3hm    2
xp3le    2
Length: 730, dtype: int64


In [76]:
# Get the base names involved in any collision
collision_base_names = set(collisions.index)

# Check whether any of YOUR 93 resolved cell lines used one of these
# collision-prone base names
your_resolved_norm_names = set(found_via_cellosaurus["cellosaurus_cell_line_name"].apply(normalise))

at_risk = your_resolved_norm_names & collision_base_names
print(f"Of your 93 resolved via Cellosaurus, names with collision risk: {len(at_risk)}")
print(at_risk)

Of your 93 resolved via Cellosaurus, names with collision risk: 0
set()


In [77]:
def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower().strip() if match else None

def normalise_base(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

# Build both fields on both sides
hpa["base_norm"] = hpa["cell line"].apply(normalise_base)
hpa["bracket_content"] = hpa["cell line"].apply(get_bracket_content)

cellosaurus["base_norm"] = cellosaurus["cellosaurus_cell_line_name"].apply(normalise_base)
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

# Match on base name AND bracket content together where bracket content exists,
# fall back to base name alone only when there's no bracket on either side
def safe_match(row, cellosaurus_df):
    candidates = cellosaurus_df[cellosaurus_df["base_norm"] == row["base_norm"]]
    if len(candidates) <= 1:
        return candidates  # no ambiguity, safe to use as-is
    if row["bracket_content"] is not None:
        refined = candidates[candidates["bracket_content"] == row["bracket_content"]]
        if len(refined) >= 1:
            return refined
    return pd.DataFrame()  # genuinely ambiguous, can't safely resolve — log it

In [20]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
print(hpa_desc.columns.tolist())

['cell line', 'disease', 'disease subtype', 'cellosaurus id', 'patient', 'primary/metastasis', 'sample collection site']


In [21]:
# Search for likely candidates rather than assuming
for col in hpa_desc.columns:
    if "cell" in col or "cvcl" in col.lower() or "cellosaurus" in col.lower():
        print(col, "→ sample:", hpa_desc[col].dropna().iloc[:3].tolist())

cell line → sample: ['143b', '22rv1', '23132/87']
cellosaurus id → sample: ['cvcl_2270', 'cvcl_1045', 'cvcl_1046']


In [22]:
test_name = "23132/87"
test_norm = normalise(test_name)
print(f"'{test_name}' normalises to: '{test_norm}'")

# Does it match anything in sample_info?
match = sample_info[sample_info["cln_norm"] == test_norm]  # or whichever norm col you used
print(match)

# Does it match anything in hpa_desc?
match2 = hpa_desc[hpa_desc["cln_norm"] == test_norm]
print(match2)

'23132/87' normalises to: '2313287'
       depmap_id cell_line_name stripped_cell_line_name        ccle_name  \
1241  ach-000948       23132/87                 2313287  2313287_stomach   

     alias  cosmicid   sex source       rrid  wtsi_master_cell_id  ...  \
1241  none  910924.0  male   dsmz  cvcl_1046                558.0  ...   

     model_manipulation model_manipulation_details patient_id  \
1241               none                       none  pt-vdirwk   

     parent_depmap_id cellosaurus_ncit_disease cellosaurus_ncit_id  \
1241             none   gastric adenocarcinoma               c4004   

     cellosaurus_issues cln_norm scln_norm  rrid_norm  
1241               none  2313287   2313287  cvcl_1046  

[1 rows x 32 columns]


KeyError: 'cln_norm'

### Cell line key resolution for hpa_rna, using hpa_desc

In [23]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")

def normalise(name):
    if pd.isna(name):
        return None
    return re.sub(r"[-/_\s]", "", str(name).lower())

hpa["cell_line_norm"] = hpa["cell line"].apply(normalise)
sample_info["cln_norm"]  = sample_info["cell_line_name"].apply(normalise)
sample_info["scln_norm"] = sample_info["stripped_cell_line_name"].apply(normalise)

# Tier 1 — direct name match
tier1 = hpa["cell_line_norm"].isin(sample_info["cln_norm"])
print(f"Tier 1 (direct name): {hpa.loc[tier1, 'cell line'].nunique()} unique cell lines")

# Tier 2 — stripped name match, only for what tier 1 missed
remaining = hpa[~tier1]
tier2 = remaining["cell_line_norm"].isin(sample_info["scln_norm"])
print(f"Tier 2 (stripped name): {remaining.loc[tier2, 'cell line'].nunique()} additional")

# Tier 3 — go through hpa_desc to get CVCL, then match sample_info.rrid
still_remaining = remaining[~tier2]
hpa_desc["cln_norm"] = hpa_desc["cell line"].apply(normalise)  # confirm actual col name
sample_info["rrid_norm"] = sample_info["rrid"]  # already lowercase cvcl_xxxx format

merged = still_remaining.merge(
    hpa_desc[["cln_norm", "cellosaurus id"]],  # confirm actual col name from your data
    on="cln_norm", how="left"
)
tier3 = merged["cellosaurus id"].isin(sample_info["rrid_norm"])
print(f"Tier 3 (via hpa_desc CVCL): {merged.loc[tier3, 'cell line'].nunique()} additional")

unresolved = merged[~tier3]
print(f"Permanently unresolved: {unresolved['cell line'].nunique()}")

Tier 1 (direct name): 1014 unique cell lines
Tier 2 (stripped name): 35 additional


KeyError: 'cln_norm'

In [18]:
sample_info.head(5)

,depmap_id,cell_line_name,stripped_cell_line_name,ccle_name,alias,cosmicid,sex,source,rrid,wtsi_master_cell_id,...,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,cellosaurus_ncit_disease,cellosaurus_ncit_id,cellosaurus_issues,cln_norm,scln_norm,rrid_norm
0,ach-000016,slr 21,slr21,slr21_kidney,none,NaN,none,academic lab,cvcl_v607,NaN,...,none,none,pt-jnarlb,none,clear cell renal cell carcinoma,c4033,none,slr21,slr21,cvcl_v607
1,ach-000032,mhh-call-3,mhhcall3,mhhcall3_haematopoietic_and_lymphoid_tissue,none,NaN,female,dsmz,cvcl_0089,NaN,...,none,none,pt-p2koyi,none,childhood b acute lymphoblastic leukemia,c9140,none,mhhcall3,mhhcall3,cvcl_0089
2,ach-000033,nci-h1819,ncih1819,ncih1819_lung,none,NaN,female,academic lab,cvcl_1497,NaN,...,none,none,pt-9p1wqv,none,lung adenocarcinoma,c3512,none,ncih1819,ncih1819,cvcl_1497
3,ach-000043,hs 895.t,hs895t,hs895t_fibroblast,none,NaN,female,atcc,cvcl_0993,NaN,...,none,none,pt-rtuvzq,none,melanoma,c3224,none,hs895.t,hs895t,cvcl_0993
4,ach-000049,hek te,hekte,hekte_kidney,none,NaN,none,academic lab,cvcl_ws59,NaN,...,immortalized,none,pt-qwyygr,none,none,none,no information is available about this cell li...,hekte,hekte,cvcl_ws59
